# 04 — Boolean Masking & Filtering

Boolean masking is the foundation of all data filtering in Pandas.
This notebook covers compound conditions, `query()`, `eval()`, IQR outlier removal,
and every interview-relevant pattern.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

employees = pd.DataFrame({
    'emp_id':           range(1001, 1001 + n),
    'name':             [f'Employee_{i:03d}' for i in range(n)],
    'department':       np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'salary':           np.random.randint(40000, 150000, n),
    'experience_years': np.random.randint(1, 20, n),
    'join_date':        pd.date_range('2010-01-01', periods=n, freq='W'),
    'rating':           np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n),
    'city':             np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'active':           np.random.choice([True, False], n, p=[0.85, 0.15])
})

# Introduce some NaN for realistic filtering scenarios
nan_mask = np.random.choice([True, False], n, p=[0.05, 0.95])
employees.loc[nan_mask, 'salary'] = np.nan

employees.head()

## 1. Creating Boolean Masks

In [ ]:
# Basic comparison — returns boolean Series
is_engineer = employees['department'] == 'Engineering'
is_high_pay = employees['salary'] > 100000

print(f"Engineers:          {is_engineer.sum()}")
print(f"High earners:       {is_high_pay.sum()}")
print(f"dtype:              {is_engineer.dtype}")

## 2. Compound Conditions — &, |, ~

**Must use parentheses** around each condition — operator precedence bites hard.

In [ ]:
# AND: both conditions true
senior_engineers = employees[
    (employees['department'] == 'Engineering') &
    (employees['experience_years'] >= 10)
]
print(f"Senior engineers: {len(senior_engineers)}")

# OR: either condition true
eng_or_finance = employees[
    (employees['department'] == 'Engineering') |
    (employees['department'] == 'Finance')
]
print(f"Eng or Finance: {len(eng_or_finance)}")

# NOT: invert mask
not_sales = employees[~(employees['department'] == 'Sales')]
print(f"Not Sales: {len(not_sales)}")

In [ ]:
# COMMON MISTAKE: using Python 'and' / 'or' instead of & / |
try:
    result = employees[
        employees['department'] == 'Engineering'
        and  # <-- WRONG! Raises ValueError
        employees['salary'] > 100000
    ]
except ValueError as e:
    print(f"ValueError: {e}")
    print("Use & instead of 'and', | instead of 'or'")

In [ ]:
# MISSING PARENTHESES mistake
# employees[employees['dept'] == 'Eng' & employees['salary'] > 100000]  # WRONG
# & has higher precedence than == so this becomes:
# employees[employees['dept'] == ('Eng' & employees['salary'] > 100000)]

# Always wrap each condition in parentheses!
correct = employees[
    (employees['department'] == 'Engineering') &
    (employees['salary'] > 100000)
]
print(f"Correct result: {len(correct)} rows")

## 3. isin() — Multiple Value Membership

In [ ]:
# Cleaner than chaining multiple OR conditions
target_depts = ['Engineering', 'Finance']
mask = employees['department'].isin(target_depts)
print(employees[mask]['department'].value_counts())

# Negate: employees NOT in those departments
print(employees[~mask]['department'].value_counts())

## 4. between() — Range Filtering

In [ ]:
# between(lo, hi) is inclusive on both ends by default
mid_experience = employees[employees['experience_years'].between(5, 10)]
print(f"5-10 years exp: {len(mid_experience)}")
print(mid_experience['experience_years'].describe())

## 5. String Filtering with str Accessor

In [ ]:
# str.contains() — regex by default
south_india = employees[employees['city'].str.contains('Bangalore|Chennai', regex=True)]
print(south_india['city'].value_counts())

# str.startswith / endswith
m_cities = employees[employees['city'].str.startswith('M')]
print(m_cities['city'].value_counts())

In [ ]:
# Handle NaN in string columns: na parameter
city_series = pd.Series(['Mumbai', None, 'Bangalore', 'Mumbai', None])

# Default: na=False returns False for NaN rows
print(city_series.str.contains('Mumbai', na=False))

## 6. NaN Filtering — isna(), notna()

In [ ]:
# Filter rows where salary is NaN
missing_salary = employees[employees['salary'].isna()]
print(f"Employees with missing salary: {len(missing_salary)}")

# Filter rows where salary is NOT NaN
valid_salary = employees[employees['salary'].notna()]
print(f"Employees with valid salary:   {len(valid_salary)}")

# Rows where ANY column is NaN
any_missing = employees[employees.isna().any(axis=1)]
print(f"Rows with any NaN:             {len(any_missing)}")

## 7. query() — SQL-Like Filtering

In [ ]:
# query() uses string expressions — cleaner syntax for complex conditions
result = employees.query("department == 'Engineering' and salary > 100000")
print(f"Query result: {len(result)} rows")

In [ ]:
# Local variable reference with @
salary_threshold = 90000
target_depts = ['Engineering', 'Finance']

result = employees.query("salary > @salary_threshold and department in @target_depts")
print(f"Filtered: {len(result)} rows")
print(result[['name', 'department', 'salary']].head())

In [ ]:
# query() with complex expression
result = employees.query(
    "salary > 80000 and experience_years >= 5 and rating >= 4.0 and active == True"
)
print(f"High performers: {len(result)}")

## 8. eval() — Create Columns Efficiently

In [ ]:
# eval() avoids creating intermediate arrays — more memory efficient
valid = employees.dropna(subset=['salary']).copy()

# Create new column with eval()
valid = valid.eval("comp_per_year = salary / experience_years")
print(valid[['name', 'salary', 'experience_years', 'comp_per_year']].head())

In [ ]:
# Performance comparison: eval vs standard
import time

big = pd.DataFrame({
    'a': np.random.randn(1_000_000),
    'b': np.random.randn(1_000_000),
    'c': np.random.randn(1_000_000),
})

start = time.perf_counter()
result_std = big['a'] + big['b'] + big['c']
t_std = time.perf_counter() - start

start = time.perf_counter()
result_eval = big.eval('a + b + c')
t_eval = time.perf_counter() - start

print(f"Standard: {t_std*1000:.2f}ms")
print(f"eval():   {t_eval*1000:.2f}ms")
print("Note: eval() shows benefits for complex multi-op expressions on large frames")

## 9. np.logical_and.reduce() for Many Conditions

In [ ]:
valid = employees.dropna(subset=['salary']).copy()

# When you have many conditions in a list — reduce handles it cleanly
conditions = [
    valid['department'].isin(['Engineering', 'Finance']),
    valid['salary'] > 80000,
    valid['experience_years'] >= 5,
    valid['active'] == True,
    valid['rating'] >= 3.0
]

combined_mask = np.logical_and.reduce(conditions)
result = valid[combined_mask]
print(f"Multi-condition filter: {len(result)} rows")
print(result[['name', 'department', 'salary', 'experience_years']].head())

## 10. np.where() — Conditional Column Creation

In [ ]:
valid = employees.dropna(subset=['salary']).copy()

# np.where(condition, value_if_true, value_if_false)
valid['salary_tier'] = np.where(valid['salary'] >= 100000, 'High', 'Standard')
print(valid['salary_tier'].value_counts())

In [ ]:
# Nested np.where for 3+ categories
valid['band'] = np.where(
    valid['salary'] >= 120000, 'Premium',
    np.where(valid['salary'] >= 80000, 'Standard', 'Entry')
)
print(valid['band'].value_counts())

## 11. pd.cut() and pd.qcut() — Binning

In [ ]:
valid = employees.dropna(subset=['salary']).copy()

# cut(): define bins by value boundaries
valid['salary_bucket'] = pd.cut(
    valid['salary'],
    bins=[0, 60000, 90000, 120000, 200000],
    labels=['Entry', 'Mid', 'Senior', 'Executive'],
    right=True    # intervals are (lo, hi] by default
)
print(valid['salary_bucket'].value_counts().sort_index())

In [ ]:
# qcut(): define bins by quantiles (equal-frequency)
valid['salary_quartile'] = pd.qcut(
    valid['salary'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)
print(valid['salary_quartile'].value_counts().sort_index())
# Each bucket has roughly equal number of employees

## 12. IQR Outlier Removal — Production Pattern

In [ ]:
valid = employees.dropna(subset=['salary']).copy()

# Interquartile Range method
Q1 = valid['salary'].quantile(0.25)
Q3 = valid['salary'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Q1: {Q1:,.0f}  Q3: {Q3:,.0f}  IQR: {IQR:,.0f}")
print(f"Lower fence: {lower_fence:,.0f}")
print(f"Upper fence: {upper_fence:,.0f}")

# Filter outliers
clean_salary = valid[valid['salary'].between(lower_fence, upper_fence)]

print(f"\nBefore: {len(valid)} rows")
print(f"After:  {len(clean_salary)} rows")
print(f"Removed {len(valid) - len(clean_salary)} outliers")

## 13. Interview: Filter Rows in Top 10%

In [ ]:
valid = employees.dropna(subset=['salary']).copy()

# Approach 1: quantile threshold
top_10_threshold = valid['salary'].quantile(0.90)
top_10 = valid[valid['salary'] >= top_10_threshold]
print(f"Top 10% threshold: {top_10_threshold:,.0f}")
print(f"Count: {len(top_10)}")

# Approach 2: rank-based
valid['rank_pct'] = valid['salary'].rank(pct=True)
top_10_rank = valid[valid['rank_pct'] >= 0.90]
print(f"\nRank-based top 10%: {len(top_10_rank)}")

## 14. df.where() — Keep Shape, Replace Non-matching

In [ ]:
valid = employees.dropna(subset=['salary']).copy()

# df.where(cond): keeps values WHERE condition is True, NaN elsewhere
# OPPOSITE of boolean indexing (which REMOVES non-matching rows)

# Keep salary values only for Engineering; NaN for others
eng_only_salaries = valid['salary'].where(valid['department'] == 'Engineering')
print("Engineering salaries (others NaN):")
print(eng_only_salaries.head(10))
print(f"\nNaN count: {eng_only_salaries.isna().sum()}")

In [ ]:
# where() with other= parameter — replace non-matching with a value
adjusted = valid['salary'].where(
    valid['department'] == 'Engineering',
    other=valid['salary'] * 0.9  # 10% less for other departments
)
print("Original salary mean:", valid['salary'].mean().round(0))
print("Adjusted salary mean:", adjusted.mean().round(0))

## 15. Performance: Boolean Mask vs query()

In [ ]:
import time

# Large DataFrame
big = pd.DataFrame({
    'salary': np.random.randint(40000, 150000, 500_000),
    'dept':   np.random.choice(['Engineering', 'Sales', 'HR'], 500_000),
    'exp':    np.random.randint(1, 20, 500_000)
})

# Boolean mask
start = time.perf_counter()
r1 = big[(big['salary'] > 100000) & (big['dept'] == 'Engineering') & (big['exp'] >= 10)]
t_mask = time.perf_counter() - start

# query()
start = time.perf_counter()
r2 = big.query("salary > 100000 and dept == 'Engineering' and exp >= 10")
t_query = time.perf_counter() - start

print(f"Boolean mask: {t_mask*1000:.1f}ms | rows: {len(r1)}")
print(f"query():      {t_query*1000:.1f}ms | rows: {len(r2)}")
print("query() uses numexpr engine when available — can be faster on large frames")

## 16. Quick Reference — Filtering Cheatsheet

| Pattern | Code |
|---------|------|
| Equality | `df[df['col'] == val]` |
| Compound AND | `df[(cond1) & (cond2)]` |
| Compound OR | `df[(cond1) \| (cond2)]` |
| NOT | `df[~cond]` |
| Membership | `df[df['col'].isin(list)]` |
| Range | `df[df['col'].between(lo, hi)]` |
| String match | `df[df['col'].str.contains('pat')]` |
| NaN rows | `df[df['col'].isna()]` |
| Top N% | `df[df['col'] >= df['col'].quantile(0.9)]` |
| SQL-like | `df.query("col > val and col2 == 'x'")` |
| IQR outliers | `df[df['col'].between(Q1-1.5*IQR, Q3+1.5*IQR)]` |